# Shareability Metric

In [551]:
import pandas as pd
from pathlib import Path
import numpy as np
from ssl_shareability_metric.ssl_encoder_shareability import ssl_shareability
import torch
from torch.utils.data import TensorDataset, DataLoader
from ssl_shareability_metric.shared_encoder import SharedEncoder
from ssl_shareability_metric.whiten_and_center import WhitenAndCenter
import copy
from ssl_shareability_metric.seperate_encoder import SeperateEncoder
torch.manual_seed(42)

In [552]:
df = pd.read_csv(Path().cwd().resolve().parent / "data/Aotizhongxin.csv")

## Collapse seperate date times into 1 date time feature

In [553]:
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

## Sort by date time to ensure t and t+1 relationship

In [554]:
df = df.sort_values("datetime", ascending=True).reset_index(drop=True)

## Drop non continous features to preserve a clean cross covariance matrix M

In [555]:
df = df.drop(columns=["No", "year", "month", "day", "hour", "wd", "station", "RAIN"])

## Handle NaN values

In [556]:
df = df.dropna().reset_index(drop=True)

## Create lagged pairs

In [557]:
current = df.copy()
future = df.shift(-1)

valid_pair = (df["datetime"].shift(-1) - df["datetime"]).eq(pd.Timedelta(hours=1))

x_current = current.loc[valid_pair].reset_index(drop=True)
x_future = future.loc[valid_pair].reset_index(drop=True)

x_current = x_current.drop(columns=["datetime"])
x_future = x_future.drop(columns=["datetime"])

x_current.head()

,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,WSPM
0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,4.4
1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,4.7
2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,5.6
3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,3.1
4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,2.0


## Convert to numpy for easier tensor work

In [558]:
current_np = np.array(x_current)
future_np = np.array(x_future)

print(current_np.shape)
print(future_np.shape)

(30943, 10)
(30943, 10)


## Establish train val and test splits

In [559]:
train_pct = 0.70
val_pct = 0.10

train_current = current_np[:int(train_pct * current_np.shape[0])]
train_future = future_np[:int(train_pct * future_np.shape[0])]

val_current = current_np[int(train_pct * current_np.shape[0]):int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0])]
val_future = future_np[int(train_pct * future_np.shape[0]):int(train_pct * future_np.shape[0]) + int(val_pct * future_np.shape[0])]

test_current = current_np[int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0]):]
test_future = future_np[int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0]):]

print(train_current.shape)
print(train_future.shape)

print(val_current.shape)
print(val_future.shape)

print(test_current.shape)
print(test_future.shape)

(21660, 10)
(21660, 10)
(3094, 10)
(3094, 10)
(6189, 10)
(6189, 10)


## Whiten and center

In [560]:
current_whiten_and_center = WhitenAndCenter()
future_whiten_and_center = WhitenAndCenter()

train_current_whiten_and_center = current_whiten_and_center.fit_transform(train_current)
val_current_whiten_and_center = current_whiten_and_center.transform(val_current)
test_current_whiten_and_center = current_whiten_and_center.transform(test_current)

train_future_whiten_and_center = future_whiten_and_center.fit_transform(train_future)
val_future_whiten_and_center = future_whiten_and_center.transform(val_future)
test_future_whiten_and_center = future_whiten_and_center.transform(test_future)

## Shareability score

In [561]:
train_shareability, train_shared, train_seperate = ssl_shareability(train_current_whiten_and_center, train_future_whiten_and_center)
print(f"Train Shareability Score: {train_shareability}, Train Shared: {train_shared}, Train Seperate: {train_seperate}")

val_shareability, val_shared, val_seperate = ssl_shareability(val_current_whiten_and_center, val_future_whiten_and_center)
print(f"Val Shareability Score: {val_shareability}, Val Shared: {val_shared}, Val Seperate: {val_seperate}")

test_shareability, test_shared, test_seperate = ssl_shareability(test_current_whiten_and_center, test_future_whiten_and_center)
print(f"Test Shareability Score: {test_shareability}, Test Shared: {test_shared}, Test Seperate: {test_seperate}")

Train Shareability Score: 0.9999427343409215, Train Shared: 0.998177335180484, Train Seperate: 0.9982344997370264
Val Shareability Score: 0.9998326727215197, Val Shared: 2.281649661280393, Val Seperate: 2.282031507401933
Test Shareability Score: 0.9998065001388509, Test Shared: 2.142078599627839, Test Seperate: 2.1424931717590874


## Convert to tensors

In [562]:
train_current_tensor = torch.tensor(train_current_whiten_and_center, dtype=torch.float32)
train_future_tensor = torch.tensor(train_future_whiten_and_center, dtype=torch.float32)
val_current_tensor = torch.tensor(val_current_whiten_and_center, dtype=torch.float32)
val_future_tensor = torch.tensor(val_future_whiten_and_center, dtype=torch.float32)
test_current_tensor = torch.tensor(test_current_whiten_and_center, dtype=torch.float32)
test_future_tensor = torch.tensor(test_future_whiten_and_center, dtype=torch.float32)

vector_size = train_current_tensor.shape[1]


## Shared encoder

In [563]:
shared_encoder = SharedEncoder(vector_size=vector_size)
shared_optimizer = torch.optim.SGD(shared_encoder.parameters(), lr=1e-1)

epochs = 5000
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    shared_encoder.train()
    Z_x = shared_encoder(train_current_tensor)
    Z_y = shared_encoder(train_future_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    shared_optimizer.zero_grad()
    loss.backward()
    shared_optimizer.step()
    with torch.no_grad():
        weights = shared_encoder.shared.weight
        weights.div_(weights.norm(p=2))
             
    shared_encoder.eval()
    with torch.no_grad():
        Z_x = shared_encoder(val_current_tensor)
        Z_y = shared_encoder(val_future_tensor)
        val_loss = -torch.abs(torch.mean(Z_x * Z_y))
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(shared_encoder.state_dict())

if best_state is not None:
    shared_encoder.load_state_dict(best_state)
    
print(f"best val loss: {best_val_loss}")

epoch: 1 train: -0.37278661131858826 val: -0.8766521215438843
epoch: 2 train: -0.9008186459541321 val: -0.8815340995788574
epoch: 3 train: -0.9048036336898804 val: -0.8864424228668213
epoch: 4 train: -0.9084949493408203 val: -0.8913654685020447
epoch: 5 train: -0.9119111895561218 val: -0.8962931036949158
epoch: 6 train: -0.9150715470314026 val: -0.9012168645858765
epoch: 7 train: -0.9179942607879639 val: -0.9061294198036194
epoch: 8 train: -0.920697808265686 val: -0.9110248684883118
epoch: 9 train: -0.9231990575790405 val: -0.9158982634544373
epoch: 10 train: -0.9255150556564331 val: -0.9207457304000854
epoch: 11 train: -0.9276611804962158 val: -0.9255644679069519
epoch: 12 train: -0.9296523332595825 val: -0.9303514957427979
epoch: 13 train: -0.9315018653869629 val: -0.9351051449775696
epoch: 14 train: -0.933222770690918 val: -0.9398247599601746
epoch: 15 train: -0.9348269701004028 val: -0.9445090889930725
epoch: 16 train: -0.936324954032898 val: -0.9491572976112366
epoch: 17 train: -0

## Seperate encoders

In [564]:
seperate_encoder = SeperateEncoder(vector_size=vector_size)
seperate_optimizer = torch.optim.SGD(seperate_encoder.parameters(), lr=1e-1)

epochs = 5000
best_val_loss = float("inf")
best_state = None

for i in range(epochs):
    seperate_encoder.train()
    Z_x, Z_y = seperate_encoder(train_current_tensor, train_future_tensor)
    loss = -torch.abs(torch.mean(Z_x * Z_y))
    train_loss = loss.item()
    seperate_optimizer.zero_grad()
    loss.backward()
    seperate_optimizer.step()
    
    with torch.no_grad():
        current_weights = seperate_encoder.current.weight
        future_weights = seperate_encoder.future.weight
        
        current_weights.div_(current_weights.norm(p=2)) 
        future_weights.div_(future_weights.norm(p=2))
             
    seperate_encoder.eval()
    with torch.no_grad():
        Z_x, Z_y = seperate_encoder(val_current_tensor, val_future_tensor)
        val_loss = -torch.abs(torch.mean(Z_x * Z_y))
            
    print(f"epoch: {i+1} train: {train_loss} val: {val_loss}")   
            
            
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(seperate_encoder.state_dict())

if best_state is not None:
    seperate_encoder.load_state_dict(best_state)
    
print(f"best val loss: {best_val_loss}")

epoch: 1 train: -0.1443634331226349 val: -0.42305272817611694
epoch: 2 train: -0.6131954193115234 val: -0.4941490590572357
epoch: 3 train: -0.6887462735176086 val: -0.5490498542785645
epoch: 4 train: -0.7472096085548401 val: -0.5905993580818176
epoch: 5 train: -0.791599690914154 val: -0.6216202974319458
epoch: 6 train: -0.82489413022995 val: -0.6445987224578857
epoch: 7 train: -0.8497084975242615 val: -0.6615707278251648
epoch: 8 train: -0.8681800961494446 val: -0.6741281151771545
epoch: 9 train: -0.8819776773452759 val: -0.6834762692451477
epoch: 10 train: -0.8923631310462952 val: -0.6905081272125244
epoch: 11 train: -0.9002710580825806 val: -0.6958757638931274
epoch: 12 train: -0.9063843488693237 val: -0.700049877166748
epoch: 13 train: -0.9111958146095276 val: -0.7033674716949463
epoch: 14 train: -0.9150598049163818 val: -0.7060695290565491
epoch: 15 train: -0.9182302355766296 val: -0.7083264589309692
epoch: 16 train: -0.920888364315033 val: -0.7102584838867188
epoch: 17 train: -0.9

## Eval holdout set

In [565]:
with torch.no_grad():
    Z_x = shared_encoder(test_current_tensor)
    Z_y = shared_encoder(test_future_tensor)
    shared_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"shared loss: {shared_loss}")

with torch.no_grad():
    Z_x, Z_y = seperate_encoder(test_current_tensor, test_future_tensor)
    sep_loss = -torch.abs(torch.mean(Z_x * Z_y))
        

print(f"seperate loss: {sep_loss}")
        

shared loss: -1.1347335577011108
seperate loss: -1.2874876260757446


## Metric vs Test Results Analysis

In [566]:
test_result = shared_loss / sep_loss
print(f"Test result: {test_result}, Train shareability result: {train_shareability}, Off-By: {100 - int((train_shareability / test_result.item()) * 100)}%")
print(f"Test result: {test_result}, Test shareability result: {test_shareability}, Off-By: {100 - int((test_shareability / test_result.item()) * 100)}%")

Test result: 0.8813549280166626, Train shareability result: 0.9999427343409215, Off-By: -13%
Test result: 0.8813549280166626, Test shareability result: 0.9998065001388509, Off-By: -13%
